# Experimenting with MF-BPR and user input

historoical interactions are based on the categories of user's previously interacted locations. User inout will be explcit preferences and context singals given by the user. Builds on tuned MF-BPR model and historical reranker. 

In [1]:
import ast
import math

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv(
    "../data/processed/train60.csv",
    dtype={
        "user_id": "string",
        "gmap_id": "string"
    }
)

train_df = train_df.dropna(
    subset=["user_id", "gmap_id"]
).copy()

# Optional: positive-only training
# train_df = train_df[train_df["rating"] >= 4].copy()

train_df = train_df.drop_duplicates(
    ["user_id", "gmap_id"]
)

print("Interactions:", len(train_df))
print("Users:", train_df["user_id"].nunique())
print("Places:", train_df["gmap_id"].nunique())

Interactions: 6303944
Users: 627365
Places: 69783


In [3]:
user_indices, unique_users = pd.factorize(
    train_df["user_id"],
    sort=True
)

activity_indices, unique_activities = pd.factorize(
    train_df["gmap_id"],
    sort=True
)

user_to_index = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

activity_to_index = {
    activity_id: index
    for index, activity_id in enumerate(unique_activities)
}

index_to_activity = {
    index: activity_id
    for index, activity_id in enumerate(unique_activities)
}

num_users = len(unique_users)
num_activities = len(unique_activities)

In [4]:
user_activity_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (user_indices, activity_indices)
    ),
    shape=(num_users, num_activities),
    dtype=np.float32
)

In [5]:
bpr_model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.001,
    iterations=50,
    random_state=42
)

bpr_model.fit(
    user_activity_matrix,
    show_progress=True
)

100%|██████████| 50/50 [02:03<00:00,  2.46s/it, train_auc=94.95%, skipped=0.15%]


In [6]:
def recommend_bpr(user_id, k=10):
    user_idx = user_to_index[user_id]

    unseen_count = (
        num_activities
        - user_activity_matrix[user_idx].getnnz()
    )

    count = min(k, unseen_count)

    if count <= 0:
        return np.array([], dtype=int), np.array([])

    items, scores = bpr_model.recommend(
        userid=user_idx,
        user_items=user_activity_matrix[user_idx],
        N=count,
        filter_already_liked_items=True
    )

    return items, scores

In [7]:
def as_list(value):
    if isinstance(value, (list, tuple, set)):
        return list(value)

    if not isinstance(value, str) or not value.strip():
        return []

    value = value.strip()

    if value.startswith("["):
        parsed = ast.literal_eval(value)

        if not isinstance(parsed, list):
            raise ValueError(
                f"Expected a category list, received: {value}"
            )

        return parsed

    return [value]


activity_df = pd.read_csv(
    "../data/processed/activity_dataset.csv",
    dtype={"gmap_id": "string"}
)

for column in ["category", "activity_groups"]:
    activity_df[column] = activity_df[column].apply(as_list)

In [8]:
def combine_labels(values):
    return sorted({
        label.strip()
        for value in values
        for label in as_list(value)
        if isinstance(label, str) and label.strip()
    })


place_metadata = (
    activity_df
    .dropna(subset=["gmap_id"])
    .groupby("gmap_id", as_index=False)
    .agg({
        "name": "first",
        "category": combine_labels,
        "activity_groups": combine_labels
    })
)

assert place_metadata["gmap_id"].is_unique

In [9]:
train_with_categories = train_df.merge(
    place_metadata,
    on="gmap_id",
    how="left",
    validate="many_to_one"
)

print(
    "Interactions missing category metadata:",
    train_with_categories["category"].isna().sum()
)

Interactions missing category metadata: 0


In [10]:
def normalized_labels(value):
    return sorted({
        label.strip().casefold()
        for label in as_list(value)
        if isinstance(label, str) and label.strip()
    })


def build_historical_preferences(column):
    history = train_with_categories[
        ["user_id", column]
    ].copy()

    history[column] = history[column].apply(
        normalized_labels
    )

    history = history[
        history[column].str.len() > 0
    ].copy()

    history["weight"] = (
        1.0 / history[column].str.len()
    )

    counts = (
        history
        .explode(column)
        .groupby(["user_id", column])["weight"]
        .sum()
    )

    totals = counts.groupby(level=0).transform("sum")

    return counts / totals


historical_specific = build_historical_preferences(
    "category"
)

historical_broad = build_historical_preferences(
    "activity_groups"
)

In [11]:
def get_bpr_candidates(user_id, candidate_k=100):
    items, scores = recommend_bpr(
        user_id,
        k=candidate_k
    )

    candidates = pd.DataFrame({
        "gmap_id": [
            index_to_activity[item]
            for item in items
        ],
        "bpr_score": scores
    })

    candidates["bpr_rank"] = (
        np.arange(len(candidates)) + 1
    )

    candidates = candidates.merge(
        place_metadata,
        on="gmap_id",
        how="left",
        validate="many_to_one"
    )

    if candidates.empty:
        candidates["bpr_normalized"] = 0.0
        return candidates

    low = candidates["bpr_score"].min()
    high = candidates["bpr_score"].max()

    if high > low:
        candidates["bpr_normalized"] = (
            (candidates["bpr_score"] - low)
            / (high - low)
        )
    else:
        candidates["bpr_normalized"] = 0.0

    return candidates

In [12]:
def preference_reranker(
    user_id,
    candidates,
    explicit_preferences=None,
):
    candidates = candidates.copy()
    explicit_preferences = explicit_preferences or {}

    def get_history(profile):
        try:
            return profile.loc[user_id]
        except KeyError:
            return {}

    specific_history = get_history(historical_specific)
    broad_history = get_history(historical_broad)

    selected_specific = set(normalized_labels(
        explicit_preferences.get("specific", [])
    ))

    selected_broad = set(normalized_labels(
        explicit_preferences.get("broad", [])
    ))

    def history_score(value, preferences):
        return max(
            (
                preferences.get(label, 0.0)
                for label in normalized_labels(value)
            ),
            default=0.0
        )

    candidates["historical_specific_score"] = (
        candidates["category"].apply(
            lambda value: history_score(
                value, specific_history
            )
        )
    )

    candidates["historical_broad_score"] = (
        candidates["activity_groups"].apply(
            lambda value: history_score(
                value, broad_history
            )
        )
    )

        # Normalize historical specific score
    specific_min = candidates["historical_specific_score"].min()
    specific_max = candidates["historical_specific_score"].max()

    if specific_max > specific_min:
        candidates["historical_specific_normalized"] = (
            candidates["historical_specific_score"] - specific_min
        ) / (specific_max - specific_min)
    else:
        candidates["historical_specific_normalized"] = 0.0


    # Normalize historical broad score
    broad_min = candidates["historical_broad_score"].min()
    broad_max = candidates["historical_broad_score"].max()

    if broad_max > broad_min:
        candidates["historical_broad_normalized"] = (
            candidates["historical_broad_score"] - broad_min
        ) / (broad_max - broad_min)
    else:
        candidates["historical_broad_normalized"] = 0.0


    specific_matches = candidates["category"].apply(
        lambda value: bool(
            set(normalized_labels(value))
            & selected_specific
        )
    )

    broad_matches = candidates["activity_groups"].apply(
        lambda value: bool(
            set(normalized_labels(value))
            & selected_broad
        )
    )

    candidates["explicit_match"] = (
        specific_matches | broad_matches
    )

    candidates["explicit_score"] = np.where(
        specific_matches,
        1.0,
        np.where(broad_matches, 0.5, 0.0)
    )

    if explicit_preferences:
        # BPR + Historical + Explicit
        candidates["rerank_score"] = (
            0.8 * candidates["bpr_normalized"]
            + 0.05 * candidates["historical_specific_normalized"]
            + 0.05 * candidates["historical_broad_normalized"]
            + 0.1 * candidates["explicit_score"]
        )

    else:
        # BPR + Historical only
        candidates["rerank_score"] = (
            0.09 * candidates["bpr_normalized"]
            + 0.5 * candidates["historical_specific_normalized"]
            + 0.5 * candidates["historical_broad_normalized"]
        )

    candidates = candidates.sort_values(
        ["rerank_score", "bpr_rank"],
        ascending=[False, True]
    ).reset_index(drop=True)

    candidates["final_rank"] = (
        np.arange(len(candidates)) + 1
    )

    return candidates

## Inspecting a randomly selcted user's historical interactions

In [13]:
interaction_counts = (
    train_df
    .groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

interaction_counts.describe()

count    627365.000000
mean         10.048288
std          10.880135
min           1.000000
25%           4.000000
50%           7.000000
75%          12.000000
max         955.000000
dtype: float64

In [14]:
interaction_counts = train_df.groupby("user_id").size()

candidate_users = interaction_counts[
    interaction_counts.between(7, 10)
]

random_user = candidate_users.sample(
    n=1,
)

display(random_user)

user_id
117833478491757346228    7
dtype: int64

random user will be continuously sampled until a user with mixed-category interactions is found 

In [15]:
user_id = "113074001886266791810"

user_history = train_with_categories[
    train_with_categories["user_id"] == user_id
]

user_history[
    ["gmap_id", "name", "category"]
]

,gmap_id,name,category
4474146,0x89d99cde4ae55f7b:0x1677a13c670bd897,Theodore's Restaurant,[Family restaurant]
4474147,0x89d977d36641764b:0x1f5cdf5fab7e09fd,Knotty Pine Diner,[Diner]
4474148,0x89d9a01cf8b37b55:0x47fe68bddd6279ec,Eddie's,"[Ice cream shop, Italian restaurant, Restaurant]"
4474149,0x89d99e51068949b5:0x4591339b899840b4,Johnnie's Pier 31,[Seafood restaurant]
4474150,0x89d98525653c8efb:0x643cb2700a5ba85,Nina's | Pizzeria & Italian Restaurant,"[Caterer, Family restaurant, Italian restauran..."
4474151,0x89d99ff617fd92d9:0xbaa167425115c71f,Harpoon Eddie's,"[Bar & grill, Beach entertainment shop, Cockta..."
4474152,0x89d99ff6ad614631:0x69c0f99caa4fe598,Canal View Cafe,"[Coffee shop, Family restaurant, Italian resta..."
4474153,0x89d9f3b92aaf0aaf:0x87e862bdfff46e48,Landmark Theatre,[Performing arts theater]
4474154,0x89d99d1b8f182ea5:0x57dd76108935e560,"Zems Ice Cream, Miniature Golf & Game Room","[Dessert shop, Ice cream shop, Miniature golf ..."


In [16]:
train_with_categories = train_df.merge(
    place_metadata[
        ["gmap_id", "name", "category", "activity_groups"]
    ],
    on="gmap_id",
    how="left",
    validate="many_to_one"
)

user_history = train_with_categories[
    train_with_categories["user_id"] == user_id
]

print("Number of historical interactions:", len(user_history))

display(
    user_history[
        ["name", "category", "activity_groups"]
    ]
)

Number of historical interactions: 9


,name,category,activity_groups
4474146,Theodore's Restaurant,[Family restaurant],[Food & Drink]
4474147,Knotty Pine Diner,[Diner],[Food & Drink]
4474148,Eddie's,"[Ice cream shop, Italian restaurant, Restaurant]",[Food & Drink]
4474149,Johnnie's Pier 31,[Seafood restaurant],[Food & Drink]
4474150,Nina's | Pizzeria & Italian Restaurant,"[Caterer, Family restaurant, Italian restauran...",[Food & Drink]
4474151,Harpoon Eddie's,"[Bar & grill, Beach entertainment shop, Cockta...","[Food & Drink, Nightlife & Live Entertainment,..."
4474152,Canal View Cafe,"[Coffee shop, Family restaurant, Italian resta...",[Food & Drink]
4474153,Landmark Theatre,[Performing arts theater],[Arts & Culture]
4474154,"Zems Ice Cream, Miniature Golf & Game Room","[Dessert shop, Ice cream shop, Miniature golf ...","[Food & Drink, Games & Activities, Sports & Re..."


In [17]:
simulated_user = {
    "user_id": "113074001886266791810",
    "explicit_preferences": {
        "broad": [
            "Food & Drink",
            "Arts & Culture"
        ],
        "specific": [
            "Italian restaurant",
            "Live music venue"
        ]
    }
}

user_id = simulated_user["user_id"]

assert user_id in user_to_index, (
    "This user is not present in the training matrix."
)

In [18]:
display(
    train_with_categories.loc[
        train_with_categories["user_id"] == user_id,
        ["name", "category", "activity_groups"]
    ]
)

,name,category,activity_groups
4474146,Theodore's Restaurant,[Family restaurant],[Food & Drink]
4474147,Knotty Pine Diner,[Diner],[Food & Drink]
4474148,Eddie's,"[Ice cream shop, Italian restaurant, Restaurant]",[Food & Drink]
4474149,Johnnie's Pier 31,[Seafood restaurant],[Food & Drink]
4474150,Nina's | Pizzeria & Italian Restaurant,"[Caterer, Family restaurant, Italian restauran...",[Food & Drink]
4474151,Harpoon Eddie's,"[Bar & grill, Beach entertainment shop, Cockta...","[Food & Drink, Nightlife & Live Entertainment,..."
4474152,Canal View Cafe,"[Coffee shop, Family restaurant, Italian resta...",[Food & Drink]
4474153,Landmark Theatre,[Performing arts theater],[Arts & Culture]
4474154,"Zems Ice Cream, Miniature Golf & Game Room","[Dessert shop, Ice cream shop, Miniature golf ...","[Food & Drink, Games & Activities, Sports & Re..."


In [19]:
candidates = get_bpr_candidates(
    user_id,
    candidate_k=50
)

historical_only = preference_reranker(
    user_id,
    candidates,
    explicit_preferences={}
)

with_explicit = preference_reranker(
    user_id,
    candidates,
    explicit_preferences=(
        simulated_user["explicit_preferences"]
    )
)

comparison = with_explicit.merge(
    historical_only[
        ["gmap_id", "final_rank"]
    ].rename(columns={
        "final_rank": "historical_rank"
    }),
    on="gmap_id",
    validate="one_to_one"
)

comparison["positions_gained"] = (
    comparison["historical_rank"]
    - comparison["final_rank"]
)

display(
    comparison[
        [
            "name",
            "category",
            "activity_groups",
            "bpr_rank",
            "historical_rank",
            "final_rank",
            "positions_gained",
            "explicit_score",
            "rerank_score"
        ]
    ].head(10)
)

,name,category,activity_groups,bpr_rank,historical_rank,final_rank,positions_gained,explicit_score,rerank_score
0,Tully's Good Times Liverpool,"[American restaurant, Bar, Family restaurant, ...",[Food & Drink],1,1,1,0,0.5,0.950000
1,Chick-fil-A,"[Breakfast restaurant, Caterer, Chicken restau...",[Food & Drink],3,2,2,0,0.5,0.823862
2,Chittenango Falls State Park,"[Park, State park, Tourist attraction]","[Outdoor & Nature, Sightseeing & Attractions]",2,40,3,37,0.0,0.714367
3,The Cheesecake Factory,"[American restaurant, Dessert restaurant, Rest...",[Food & Drink],4,23,4,19,0.5,0.652925
4,Rosamond Gifford Zoo,"[Aquarium, Cafe, Event venue, Gift shop, Natur...","[Food & Drink, Outdoor & Nature, Sightseeing &...",6,32,5,27,0.5,0.577793
5,The Wild Animal Park,"[Tourist attraction, Zoo]",[Sightseeing & Attractions],5,47,6,41,0.0,0.490771
6,Flo's Diner,"[American restaurant, Diner]",[Food & Drink],8,10,7,3,0.5,0.483593
7,Enchanted Forest Water Safari,"[Amusement park, Theme park, Water park]","[Outdoor & Nature, Sightseeing & Attractions]",7,44,8,36,0.0,0.458561
8,Mother's Cupboard,"[Diner, Fish & chips restaurant, Restaurant]",[Food & Drink],11,11,9,2,0.5,0.439912
9,Dave & Buster's,"[Amusement center, Bar & grill, Restaurant, Sp...","[Food & Drink, Games & Activities]",10,24,10,14,0.5,0.437182


# Evaluation 

In [20]:
def category_match_rate(frame, preferences, k=10):
    selected_specific = set(normalized_labels(
        preferences.get("specific", [])
    ))

    selected_broad = set(normalized_labels(
        preferences.get("broad", [])
    ))

    top = frame.head(k)

    if top.empty:
        return np.nan

    matches = [
        bool(
            set(normalized_labels(categories))
            & selected_specific
        )
        or bool(
            set(normalized_labels(groups))
            & selected_broad
        )
        for categories, groups in zip(
            top["category"],
            top["activity_groups"]
        )
    ]

    return np.mean(matches)


preferences = simulated_user["explicit_preferences"]

print(
    "Historical-only category match:",
    category_match_rate(historical_only, preferences)
)

print(
    "With explicit preferences category match:",
    category_match_rate(with_explicit, preferences)
)

Historical-only category match: 1.0
With explicit preferences category match: 0.7


For this specific user, adding in  explicit preferences increased rankings slightly.

In [21]:
import math

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    # Actual DCG
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    # Best possible DCG
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    if len(hits) > 0:
        return 1
    else:
        return 0


def calculate_metrics(recommended, relevant, k=10):

    recall = recall_at_k(recommended, relevant, k)
    hit_rate = hit_rate_at_k(recommended, relevant, k)
    ndcg = ndcg_at_k(recommended, relevant, k)

    return {
        f"Recall@{k}": recall,
        f"HitRate@{k}": hit_rate,
        f"NDCG@{k}": ndcg
    }
   

In [22]:
validation_df = pd.read_csv(
    "../data/processed/validation20.csv",
    dtype={
        "user_id": "string",
        "gmap_id": "string",
        "rating": float
    }
)

sample_df = pd.read_csv(
    "../data/processed/validation_sample_1000_users.csv",
    dtype={"user_id": "string"}
)

validation_pos = validation_df[
    validation_df["rating"] >= 4
].copy()

user_id = simulated_user["user_id"]

candidates = get_bpr_candidates(
    user_id,
    candidate_k=50
)

reranked = preference_reranker(
    user_id,
    candidates,
    simulated_user["explicit_preferences"]
)

recommended = (
    reranked
    .head(10)["gmap_id"]
    .tolist()
)

relevant_by_user = (
    validation_pos
    .groupby("user_id")["gmap_id"]
    .apply(set)
    .to_dict()
)

relevant = relevant_by_user[user_id]

metrics = calculate_metrics(
    recommended,
    relevant,
    k=10
)

print(metrics)

{'Recall@10': 0.0, 'HitRate@10': 0, 'NDCG@10': 0.0}


In [23]:
preferences = simulated_user["explicit_preferences"]

print(
    "BPR category match:",
    category_match_rate(
        candidates.sort_values("bpr_rank"),
        preferences
    )
)
"""
print(
    "Historical-only category match:",
    category_match_rate(
        historical_only,
        preferences
    )
)
"""
print(
    "With explicit preferences category match:",
    category_match_rate(
        with_explicit,
        preferences
    )
)

BPR category match: 0.7
With explicit preferences category match: 0.7
